In [6]:
import numpy as np
import pandas as pd

def calculate_opponent_score(player_information):
    """
    计算每个选手的对手小分，即所有曾被该选手击败的对手的分数之和。
    """
    opponent_scores = []
    for i, row in player_information.iterrows():
        defeated_opponents = row['Defeated_Opponents']
        # 根据 defeated_opponents 列表中存放的选手编号，计算这些选手的 Score 总和
        opponent_score = player_information.loc[player_information['Player'].isin(defeated_opponents), 'Score'].sum()
        opponent_scores.append(opponent_score)
    player_information['Opponent_Score'] = opponent_scores
    return player_information

def rank_players(player_information):
    """
    对选手进行排名，按总分 -> 对手小分 -> 直接胜负关系排序 -> tie-breaker排序。
    当总分和对手小分相同时，如果直接胜负关系无法区分，则使用随机生成的 tie-breaker 值进行排序，
    避免默认使用选手编号排序。
    """
    # 计算对手小分
    player_information = calculate_opponent_score(player_information)
    
    # 为每个选手增加一个随机 tie-breaker 值
    player_information = player_information.copy()
    player_information["TieBreaker"] = np.random.rand(len(player_information))
    
    # 按总分、对手小分、tie-breaker排序（均降序排序）
    player_information = player_information.sort_values(
        by=['Score', 'Opponent_Score', 'TieBreaker'], 
        ascending=False
    ).reset_index(drop=True)
    
    # 针对总分和对手小分相同的情况，使用直接胜负关系重新调整顺序
    ranked_list = [player_information.iloc[0]]  # 添加第一位选手
    for i in range(1, len(player_information)):
        current_player = player_information.iloc[i]
        previous_player = ranked_list[-1]
        
        # 如果当前选手与前一位选手总分和对手小分相同，则检查直接胜负关系
        if (current_player['Score'] == previous_player['Score'] and 
            current_player['Opponent_Score'] == previous_player['Opponent_Score']):
            # 若当前选手曾战胜前一位选手，则应排在前面
            if current_player['Player'] in previous_player['Defeated_Opponents']:
                ranked_list.insert(len(ranked_list) - 1, current_player)
            # 若前一位选手曾战胜当前选手，则当前选手放后面
            elif previous_player['Player'] in current_player['Defeated_Opponents']:
                ranked_list.append(current_player)
            else:
                # 如果双方没有直接对阵，则依赖 tie-breaker 保持原排序
                ranked_list.append(current_player)
        else:
            ranked_list.append(current_player)
    
    ranked_df = pd.DataFrame(ranked_list)
    return ranked_df

def test_rank_players_complex():
    # 构造复杂的测试数据：12 个选手，不同的总分和丰富的直接胜负记录
    data = {
        "Player": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
        "Score":  [1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0],
        # 直接胜负记录说明：
        # 选手1曾战胜2、3
        # 选手2曾战胜4、5
        # 选手3曾战胜6、7
        # 选手4曾战胜1（与选手1形成循环）
        # 选手5曾战胜7、8、9
        # 选手6曾战胜10
        # 选手7曾战胜11
        # 选手8曾战胜2（形成部分循环）
        # 选手9无直接胜负记录
        # 选手10曾战胜11、12
        # 选手11曾战胜?（这里只保留被战胜记录，可为空或其他情况，这里设定为空）
        # 选手12曾战胜1、3（形成环状关系，与选手1和选手3有对抗）
        "Defeated_Opponents": [
            [2, 3],     # 选手1
            [4, 5],     # 选手2
            [6, 7],     # 选手3
            [1],        # 选手4
            [7, 8, 9],  # 选手5
            [10],       # 选手6
            [11],       # 选手7
            [2],        # 选手8
            [],         # 选手9
            [11, 12],   # 选手10
            [],         # 选手11
            [1, 3]      # 选手12
        ]
    }
    player_information = pd.DataFrame(data)
    
    print("原始选手信息:")
    print(player_information)
    
    ranked_df = rank_players(player_information)
    
    print("\n排名后的选手信息:")
    print(ranked_df)

test_rank_players_complex()

原始选手信息:
    Player  Score Defeated_Opponents
0        1      1             [2, 3]
1        2      1             [4, 5]
2        3      0             [6, 7]
3        4      1                [1]
4        5      1          [7, 8, 9]
5        6      0               [10]
6        7      1               [11]
7        8      0                [2]
8        9      0                 []
9       10      1           [11, 12]
10      11      0                 []
11      12      0             [1, 3]

排名后的选手信息:
    Player  Score Defeated_Opponents  Opponent_Score  TieBreaker
0        2      1             [4, 5]               2    0.168214
1        1      1             [2, 3]               1    0.821099
2        4      1                [1]               1    0.430771
3        5      1          [7, 8, 9]               1    0.233592
4       10      1           [11, 12]               0    0.431177
5        7      1               [11]               0    0.114614
6        3      0             [6, 7]         